# LPPR: PS2-Style Learnable PPR — Planetoid (Cora / CiteSeer / PubMed)

**Architecture**: Soft PPR adjacency within extracted subgraphs, with a learned per-edge scale selector.

- **Selector (θ)**: `PPRScaleSelector` — takes PPR-diffused cross-pair features → per-edge alpha weights
- **Encoder (w)**: `PPRDiffusionEncoder` — GNN with `P_soft = Σ wᵢ · PPR_αᵢ|_S` as adjacency
- **Bi-level**: θ trained on val loss, w trained on train loss
- **Subgraphs**: score-threshold PPR extraction (no full-graph pass at any stage)

Inspired by [PS2 (WSDM'23)](https://github.com/qiaoyu-tan/PS2). Results saved to `results/benchmark-option-a/`.

In [ ]:
# PPR scale candidates: restart probabilities (high=local, low=global)
# Empirically from exploration at τ=1e-3:
#   0.90 → ~12 nodes (local, ~k=1 depth),  0.50 → ~42 nodes (~k=1.5),  0.15 → ~98 nodes (~k=2 size)
TELEPORT_VALUES  = [0.90, 0.50, 0.15]
PUSH_EPSILON        = 1e-5   # precise PPR — used ONLY for the pre-built train cache
SEARCH_PUSH_EPSILON = 5e-4   # ~20x faster — for all live extractions during
                             # search and fine-tuning (negatives, val subgraphs)
SCORE_TAU        = 1e-3      # subgraph node threshold
EXTRACTION_ALPHA = 0.15      # widest scale → largest envelope subgraph

DATASETS = ['Cora', 'CiteSeer', 'PubMed']

# Model
HIDDEN_CHANNELS  = 256
NUM_LAYERS       = 3
DROPOUT          = 0.3
SELECTOR_HIDDEN  = 256
SELECTOR_LAYERS  = 3

# Phase 1: Architecture Search (bi-level DARTS)
SEARCH_EPOCHS          = 50
SEARCH_BATCH_SIZE      = 32       # small — each batch is B subgraph GNN forwards
SEARCH_LR              = 0.01
SEARCH_LR_SELECTOR     = 3e-4
SEARCH_LR_MIN          = 1e-4
TEMPERATURE_START      = 1.0
TEMPERATURE_END        = 0.2
EDGES_PER_SEARCH_EPOCH = None     # None = all edges (small Planetoid graphs)
SEARCH_VAL_EVERY       = 5

# Phase 2: Fine-tune with frozen selector
FINETUNE_EPOCHS     = 200
FINETUNE_BATCH_SIZE = 32
FINETUNE_LR         = 0.005
FINETUNE_PATIENCE   = 30
FINETUNE_EVAL_STEPS = 5
WEIGHT_DECAY        = 1e-5
GRAD_CLIP           = 1.0

SEED = 42

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Teleport values: {TELEPORT_VALUES}  ({len(TELEPORT_VALUES)} scales)')
print(f'Datasets: {DATASETS}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, time, copy
import pandas as pd

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
import random; random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sns.set_theme(style='whitegrid', palette='muted')

from subgrapher.benchmark.data_prep import prepare_planetoid_data
from subgrapher.benchmark_learnable_ppr.multi_scale_ppr import MultiScalePPR
from subgrapher.benchmark_learnable_ppr.option_a_model import LPPRGNN, PPRScaleSelector
from subgrapher.benchmark_learnable_ppr.option_a_extractor import (
    LPPRSubgraphExtractor, build_or_load_cache, sample_neg_subgraphs)
from subgrapher.benchmark_learnable_ppr.option_a_searcher import LPPRSearcher
from subgrapher.benchmark_learnable_ppr.run_option_a_benchmark import (
    finetune_lppr, evaluate_lppr_per_subgraph)


def ensure_train_only_edges(data, dd):
    if not getattr(data, '_edge_index_train_only', False):
        data._orig_edge_index = data.edge_index
        data.edge_index = dd['train_edge_index']
        data._edge_index_train_only = True
        print(f'  [train-only edges] swapped: '
              f"{data._orig_edge_index.size(1):,} -> {data.edge_index.size(1):,}")
    return data

## 1. Data Loading and Multi-Scale PPR

In [ ]:
all_results = {}

for dataset_name in DATASETS:
    print(f'\n{"=" * 60}')
    print(f'Dataset: {dataset_name}')
    print(f'{"=" * 60}')

    dd = prepare_planetoid_data(dataset_name)
    data = dd['data']
    split_edge = dd['split_edge']
    feature_dim = dd['feature_dim']

    print(f'Nodes: {data.num_nodes:,}  Edges: {data.edge_index.size(1):,}  '
          f'Feature dim: {feature_dim}')
    print(f'Train: {split_edge["train"]["source_node"].size(0):,}  '
          f'Val: {split_edge["valid"]["source_node"].size(0):,}  '
          f'Test: {split_edge["test"]["source_node"].size(0):,}')

    ensure_train_only_edges(data, dd)

    print(f'\nLoading multi-scale PPR (teleport={TELEPORT_VALUES})...')
    multi_scale_ppr = MultiScalePPR(
        dataset_name, data, teleport_values=TELEPORT_VALUES, device=DEVICE)
    print(multi_scale_ppr)

    all_results[dataset_name] = {
        'data': data, 'split_edge': split_edge,
        'dd': dd, 'feature_dim': feature_dim,
        'multi_scale_ppr': multi_scale_ppr, 'experiments': {},
    }

## 2. Subgraph Extractor and Caches

Extracts score-threshold PPR subgraphs: run push from seeds {u, v}, keep nodes where absorbed_score > τ.\
Uses the widest scale (α=0.25) for extraction so the envelope is large enough for all selector choices.\
**One-time cost** — cached to disk for reuse.

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']
    ensure_train_only_edges(data, r['dd'])

    # Precise extractor — used to build the train cache (one-time cost)
    extractor = LPPRSubgraphExtractor(
        data,
        push_epsilon=PUSH_EPSILON,
        score_tau=SCORE_TAU,
        extraction_alpha=EXTRACTION_ALPHA)
    r['extractor'] = extractor

    # Fast extractor — used for all LIVE extractions during search & fine-tune
    # (train negatives, val positives, val negatives)
    search_extractor = LPPRSubgraphExtractor(
        data,
        push_epsilon=SEARCH_PUSH_EPSILON,
        score_tau=SCORE_TAU,
        extraction_alpha=EXTRACTION_ALPHA)
    r['search_extractor'] = search_extractor

    cache_dir = f'cache/option-a/{dataset_name}/eps{PUSH_EPSILON}_tau{SCORE_TAU}'
    r['cache_dir'] = cache_dir

    print(f'\n[Cache] {dataset_name}')
    train_cache = build_or_load_cache(
        extractor, split_edge, 'train', cache_dir, verbose=True)
    r['train_cache'] = train_cache
    print(f'  Train cache: {len(train_cache)} subgraphs')

print('\nAll caches ready.')

## 3. Phase 1: Bi-Level Architecture Search

**Inner loop (w)**: `PPRDiffusionEncoder + LinkPredictor` trained on train loss\
**Outer loop (θ)**: `PPRScaleSelector` trained on val loss\

Temperature annealing drives the selector toward a hard discrete choice.

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']
    msp = r['multi_scale_ppr']; feature_dim = r['feature_dim']
    extractor = r['extractor']
    search_extractor = r['search_extractor']
    train_cache = r['train_cache']
    ensure_train_only_edges(data, r['dd'])

    print(f'\n{"=" * 60}')
    print(f'Phase 1 Search: {dataset_name}  (feat_dim={feature_dim})')
    print(f'{"=" * 60}')

    model = LPPRGNN(
        feat_dim=feature_dim,
        hidden_channels=HIDDEN_CHANNELS,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        alphas=TELEPORT_VALUES,
        selector_hidden=SELECTOR_HIDDEN,
        selector_layers=SELECTOR_LAYERS)

    selector = PPRScaleSelector(
        in_channels=feature_dim,
        hidden_channels=SELECTOR_HIDDEN,
        num_layers=SELECTOR_LAYERS,
        num_scales=len(TELEPORT_VALUES),
        temperature=TEMPERATURE_START)

    print(f'  LPPRGNN params:  {sum(p.numel() for p in model.parameters()):,}')
    print(f'  Selector params: {sum(p.numel() for p in selector.parameters()):,}')

    searcher = LPPRSearcher(
        model=model,
        selector=selector,
        multi_scale_ppr=msp,
        data=data,
        split_edge=split_edge,
        extractor=extractor,
        train_cache=train_cache,
        device=DEVICE,
        lr=SEARCH_LR,
        lr_selector=SEARCH_LR_SELECTOR,
        lr_min=SEARCH_LR_MIN,
        temperature_start=TEMPERATURE_START,
        temperature_end=TEMPERATURE_END,
        edges_per_epoch=EDGES_PER_SEARCH_EPOCH,
        search_extractor=search_extractor)

    search_history = searcher.search(
        epochs=SEARCH_EPOCHS,
        batch_size=SEARCH_BATCH_SIZE,
        verbose=True,
        val_every=SEARCH_VAL_EVERY)

    print(f'Search done: {search_history["total_time"]:.1f}s  '
          f'best_val={search_history["best_val_loss"]:.4f} '
          f'@epoch {search_history["best_epoch"]}')

    # Alpha selection distribution on training edges
    alpha_indices = searcher.get_edge_alpha_indices('train')
    alpha_counts  = torch.zeros(len(TELEPORT_VALUES), dtype=torch.long)
    for k in range(len(TELEPORT_VALUES)):
        alpha_counts[k] = (alpha_indices == k).sum()
    dominant_idx = alpha_counts.argmax().item()
    print(f'  Alpha counts (train): {alpha_counts.tolist()}')
    print(f'  Dominant alpha: {TELEPORT_VALUES[dominant_idx]} '
          f'(~{(1-TELEPORT_VALUES[dominant_idx])/TELEPORT_VALUES[dominant_idx]:.1f}-hop)')

    r['experiments'][dataset_name] = {
        'model': model, 'selector': selector,
        'search_history': search_history,
        'alpha_counts': alpha_counts,
        'alpha_indices': alpha_indices,
    }

### 3b. Alpha Distribution After Search

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    exp = r['experiments'][dataset_name]
    counts = exp['alpha_counts'].numpy().astype(float)
    total  = counts.sum()
    pcts   = 100 * counts / max(total, 1)
    labels = [f'α={tv}\n(~{(1-tv)/tv:.1f}-hop)' for tv in TELEPORT_VALUES]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(labels, pcts, color=sns.color_palette('muted', len(TELEPORT_VALUES)))
    for bar, pct, cnt in zip(bars, pcts, counts):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f'{int(cnt)}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('% of training edges')
    ax.set_title(f'{dataset_name} — Learned Alpha Distribution (Train)',
                 fontsize=13, fontweight='bold')
    ax.set_ylim(0, 110)
    plt.tight_layout(); plt.show()

### 3c. Phase 1 Diagnostic Curves

In [ ]:
import math

for dataset_name in DATASETS:
    r = all_results[dataset_name]
    hist = r['experiments'][dataset_name]['search_history']
    n_ep = len(hist['temperature'])
    epochs_list = list(range(1, n_ep + 1))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss curves
    ax = axes[0]
    ax.plot(epochs_list, hist['search_loss'], label='Train Loss')
    val_epochs = [e for e, v in zip(epochs_list, hist['val_loss']) if v is not None]
    val_vals   = [v for v in hist['val_loss'] if v is not None]
    if val_vals:
        ax.plot(val_epochs, val_vals, marker='.', label='Val Loss')
    ax.axhline(hist['best_val_loss'], color='red', linestyle='--', alpha=0.5,
               label=f'Best val: {hist["best_val_loss"]:.4f} @ep{hist["best_epoch"]}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('Search Loss'); ax.legend()

    # Temperature + entropy
    ax1 = axes[1]
    ax1.plot(epochs_list, hist['temperature'], color='tab:blue', label='Temperature')
    ax1.set_ylabel('Temperature', color='tab:blue')
    ax2 = ax1.twinx()
    ent = [v if not (isinstance(v, float) and math.isnan(v)) else None
           for v in hist['arch_entropy']]
    ax2.plot(epochs_list, ent, color='tab:orange', label='Entropy')
    top1 = [v if not (isinstance(v, float) and math.isnan(v)) else None
            for v in hist['top1_mass']]
    ax2.plot(epochs_list, top1, color='tab:green', linestyle='--', label='Top-1 mass')
    lines1, lab1 = ax1.get_legend_handles_labels()
    lines2, lab2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, lab1 + lab2, loc='center right')
    ax1.set_title('Temperature + Selector Entropy')

    # Summary
    ax = axes[2]
    exp = r['experiments'][dataset_name]
    counts = exp['alpha_counts']
    top_pct = 100 * counts.max().item() / max(counts.sum().item(), 1)
    dom_alpha = TELEPORT_VALUES[counts.argmax().item()]
    lines = [
        f'Dominant α: {dom_alpha} ({top_pct:.1f}%)',
        f'Best val loss: {hist["best_val_loss"]:.4f}',
        f'Best epoch: {hist["best_epoch"]}',
        f'Search time: {hist["total_time"]:.0f}s',
    ]
    if top_pct > 95:
        lines.insert(0, 'WARNING: ALPHA COLLAPSE (>95%)')
        ax.set_facecolor('#fff0f0')
    else:
        lines.insert(0, 'Status: OK')
        ax.set_facecolor('#f0fff0')
    ax.text(0.5, 0.5, '\n'.join(lines), transform=ax.transAxes,
            ha='center', va='center', fontsize=12, fontfamily='monospace')
    ax.set_title('Summary'); ax.axis('off')

    fig.suptitle(f'{dataset_name} — Phase 1 Diagnostics',
                 fontsize=15, fontweight='bold')
    plt.tight_layout(); plt.show()

## 4. Phase 2: Fine-Tune with Frozen Selector

Selector is set to eval mode (hard argmax). Only the PPR-diffusion encoder and link predictor weights are updated.

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']
    msp = r['multi_scale_ppr']
    extractor = r['extractor']
    search_extractor = r['search_extractor']
    train_cache = r['train_cache']
    ensure_train_only_edges(data, r['dd'])

    exp = r['experiments'][dataset_name]
    model    = exp['model']
    selector = exp['selector']

    print(f'\n{"=" * 60}')
    print(f'Phase 2 Fine-tune: {dataset_name}')
    print(f'{"=" * 60}')

    ckpt_dir = f'checkpoints/option-a/{dataset_name}'
    exp['ckpt_dir'] = ckpt_dir

    ft_history = finetune_lppr(
        model=model,
        selector=selector,
        extractor=extractor,
        neg_extractor=search_extractor,
        multi_scale_ppr=msp,
        data=data,
        split_edge=split_edge,
        train_cache=train_cache,
        epochs=FINETUNE_EPOCHS,
        batch_size=FINETUNE_BATCH_SIZE,
        lr=FINETUNE_LR,
        weight_decay=WEIGHT_DECAY,
        grad_clip=GRAD_CLIP,
        patience=FINETUNE_PATIENCE,
        eval_steps=FINETUNE_EVAL_STEPS,
        device=DEVICE,
        verbose=True,
        checkpoint_dir=ckpt_dir)

    exp['ft_history'] = ft_history
    print(f'Best val loss: {ft_history["best_val_loss"]:.4f} '
          f'@epoch {ft_history["best_epoch"]}  '
          f'Early stop: {ft_history["stopped_early"]}')

### 4b. Fine-Tuning Loss Curves

In [ ]:
for dataset_name in DATASETS:
    exp = all_results[dataset_name]['experiments'][dataset_name]
    hist = exp.get('ft_history', {})
    if not hist: continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ep = list(range(1, len(hist['train_loss']) + 1))

    axes[0].plot(ep, hist['train_loss'], label='Train Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Train Loss'); axes[0].legend()

    if hist['val_loss']:
        val_steps = list(range(FINETUNE_EVAL_STEPS,
                               FINETUNE_EVAL_STEPS * (len(hist['val_loss']) + 1),
                               FINETUNE_EVAL_STEPS))
        axes[1].plot(val_steps, hist['val_loss'], marker='o', label='Val Loss')
        axes[1].axhline(hist['best_val_loss'], color='red', linestyle='--', alpha=0.5,
                        label=f'Best: {hist["best_val_loss"]:.4f}')
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
        axes[1].set_title('Val Loss'); axes[1].legend()

    fig.suptitle(f'{dataset_name} — Fine-Tuning Curves',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

## 5. Evaluation (Per-Subgraph)

For each positive (u, v): extract subgraph → get score.\
For each negative (u, neg_j): same.\
Rank the positive among 1 + num_neg negatives → MRR / Hits@K / AUC.

⚠️ **Note**: Evaluation is per-subgraph (not full-graph) — consistent with training. This is not directly comparable to full-graph baselines.

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']
    msp = r['multi_scale_ppr']; extractor = r['extractor']
    exp = r['experiments'][dataset_name]
    model = exp['model']; selector = exp['selector']

    print(f'\n{"=" * 60}')
    print(f'Evaluation: {dataset_name}')
    print(f'{"=" * 60}')

    val_results = evaluate_lppr_per_subgraph(
        model, selector, extractor, msp,
        data, split_edge, split='valid',
        batch_size=SEARCH_BATCH_SIZE, device=DEVICE)
    test_results = evaluate_lppr_per_subgraph(
        model, selector, extractor, msp,
        data, split_edge, split='test',
        batch_size=SEARCH_BATCH_SIZE, device=DEVICE)

    exp['val_results']  = val_results
    exp['test_results'] = test_results

    print(f'  Val  MRR={val_results["mrr"]:.4f}  '
          f'AUC={val_results.get("auc",0):.4f}  '
          f'Hits@10={val_results.get("hits@10",0):.4f}')
    print(f'  Test MRR={test_results["mrr"]:.4f}  '
          f'AUC={test_results.get("auc",0):.4f}  '
          f'Hits@10={test_results.get("hits@10",0):.4f}')

## 6. Cross-Method Comparison

In [ ]:
def load_baseline_results(base_dir, method, ds_filter=None):
    rows = []
    if not os.path.exists(base_dir): return rows
    for root, dirs, files in os.walk(base_dir):
        if 'runs' in dirs: dirs.remove('runs')
        if 'full_results.json' in files:
            with open(os.path.join(root, 'full_results.json')) as f:
                r = json.load(f)
            if ds_filter and r.get('dataset') not in ds_filter: continue
            rows.append(r)
    return rows

comparison_rows = []

for r in load_baseline_results('results/benchmark', 'Full Graph', DATASETS):
    comparison_rows.append({
        'Dataset': r['dataset'], 'Encoder': r['encoder'], 'Method': 'Full Graph',
        'MRR': r['test_results']['mrr'], 'AUC': r['test_results'].get('auc', 0),
    })

for r in load_baseline_results('results/benchmark-ppr', 'Static PPR', DATASETS):
    comparison_rows.append({
        'Dataset': r['dataset'], 'Encoder': r['encoder'], 'Method': 'Static PPR',
        'MRR': r['test_results']['mrr'], 'AUC': r['test_results'].get('auc', 0),
    })

for r in load_baseline_results('results/benchmark-khop', 'Static k-hop', DATASETS):
    comparison_rows.append({
        'Dataset': r['dataset'], 'Encoder': r['encoder'], 'Method': 'Static k-hop',
        'MRR': r['test_results']['mrr'], 'AUC': r['test_results'].get('auc', 0),
    })

for ds in DATASETS:
    exp = all_results.get(ds, {}).get('experiments', {}).get(ds, {})
    tr = exp.get('test_results')
    if tr:
        comparison_rows.append({
            'Dataset': ds, 'Encoder': 'PPRDiff', 'Method': 'LPPR',
            'MRR': tr['mrr'], 'AUC': tr.get('auc', 0),
        })

if comparison_rows:
    df = pd.DataFrame(comparison_rows)
    for metric in ['MRR', 'AUC']:
        fig, axes = plt.subplots(1, len(DATASETS),
                                  figsize=(6 * len(DATASETS), 5), sharey=False)
        if len(DATASETS) == 1: axes = [axes]
        for ax, ds in zip(axes, DATASETS):
            sub = df[df['Dataset'] == ds]
            sns.barplot(data=sub, x='Encoder', y=metric, hue='Method', ax=ax)
            ax.set_title(ds); ax.set_xlabel('')
            ax.legend(loc='lower right', fontsize=8)
        fig.suptitle(f'Test {metric} — All Methods', fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()
    display(df.pivot_table(index=['Dataset', 'Encoder'], columns='Method',
                           values=['MRR', 'AUC']).round(4))
else:
    print('No baseline data found — run baselines first, then re-run this cell.')

## 7. Save Results

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    exp = r['experiments'][dataset_name]
    tr = exp.get('test_results')
    if not tr: continue

    sh = exp['search_history']
    fh = exp['ft_history']
    run_id = time.strftime('%Y%m%d_%H%M%S')

    result_json = {
        'dataset': dataset_name,
        'encoder': 'PPRDiff',
        'method': 'LPPR',
        'teleport_values': TELEPORT_VALUES,
        'extraction_alpha': EXTRACTION_ALPHA,
        'push_epsilon': PUSH_EPSILON,
        'score_tau': SCORE_TAU,
        'test_results': {k: float(v) for k, v in tr.items()},
        'val_results':  {k: float(v) for k, v in exp.get('val_results', {}).items()},
        'alpha_distribution': exp['alpha_counts'].tolist(),
        'dominant_alpha': TELEPORT_VALUES[exp['alpha_counts'].argmax().item()],
        'search_time':   sh['total_time'],
        'finetune_time': fh.get('total_time', 0),
        'train_time':    sh['total_time'] + fh.get('total_time', 0),
        'best_val_loss_search':   sh['best_val_loss'],
        'best_epoch_search':      sh['best_epoch'],
        'best_val_loss_finetune': fh['best_val_loss'],
        'best_epoch_finetune':    fh['best_epoch'],
        'stopped_early': fh['stopped_early'],
        'config': {
            'feature_dim': r['feature_dim'],
            'hidden_channels': HIDDEN_CHANNELS,
            'num_layers': NUM_LAYERS,
            'dropout': DROPOUT,
            'selector_hidden': SELECTOR_HIDDEN,
            'selector_layers': SELECTOR_LAYERS,
            'search_epochs': SEARCH_EPOCHS,
            'finetune_epochs': FINETUNE_EPOCHS,
            'finetune_lr': FINETUNE_LR,
            'finetune_patience': FINETUNE_PATIENCE,
        },
        'seed': SEED,
        'run_id': run_id,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }

    save_dir = f'results/benchmark-option-a/{dataset_name}'
    run_dir  = os.path.join(save_dir, 'runs', run_id)
    os.makedirs(run_dir, exist_ok=True)

    for p in [os.path.join(run_dir, 'full_results.json'),
              os.path.join(save_dir, 'full_results.json')]:
        with open(p, 'w') as f:
            json.dump(result_json, f, indent=2)

    print(f'Saved: {save_dir}/full_results.json')
    print(f'  MRR={tr["mrr"]:.4f}  AUC={tr.get("auc",0):.4f}  '
          f'Hits@10={tr.get("hits@10",0):.4f}')

print('Done!')